# 05 · Extensiones

## Paso 9 · Rebalanceo mensual, visión desde México y panel

Todas las comparaciones usan los parámetros fijos del Paso 3, 5 bps por operación, `delay = 1` y el periodo 2006-01-13 a 2025-12-30.

In [1]:
import sys; sys.path.append("..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.data import download_prices, download_fx
from src.signals import STRATEGIES, monthly
from src.backtest import run_backtest
from src.metrics import summarize, sharpe
sns.set_theme(style="whitegrid")

px = download_prices(cache="../data/raw/prices.parquet")
start = px.index[260]
cols = ["cagr", "vol", "sharpe", "max_dd", "turnover_yr", "cost_drag_yr"]

### 9.1 Rebalanceo mensual

Las estrategias que operan a diario pagan muchos costos. Con `monthly()` cada estrategia decide solo al cierre de cada mes. No hay look-ahead: se comprueba igual que en el Paso 3. Detalle de implementación: el cambio se registra el último día hábil del mes cuando el mes termina en día hábil, o el primer día hábil del siguiente cuando termina en fin de semana; nunca a mitad de mes.

In [2]:
def check_no_lookahead(signal_fn, prices, cut=2000):
    pd.testing.assert_frame_equal(signal_fn(prices).iloc[:cut], signal_fn(prices.iloc[:cut]))

for name in ["sma_cross", "momentum", "mean_reversion", "vol_target"]:
    for cut in [300, 2000, 4000]:
        check_no_lookahead(monthly(STRATEGIES[name]), px, cut)
    print(name, "(mensual) ok")

sma_cross (mensual) ok
momentum (mensual) ok
mean_reversion (mensual) ok
vol_target (mensual) ok


In [3]:
rows = []
for name in ["sma_cross", "momentum", "mean_reversion", "vol_target"]:
    for label, fn in [("diaria", STRATEGIES[name]), ("mensual", monthly(STRATEGIES[name]))]:
        res = run_backtest(px, fn(px), cost_bps=5, start=start)
        rows.append({"estrategia": name, "frecuencia": label, **{k: v for k, v in summarize(res).items() if k in cols}})
freq = pd.DataFrame(rows).set_index(["estrategia", "frecuencia"])
freq.round(3)

cagr    vol  sharpe  max_dd  turnover_yr  \
estrategia     frecuencia                                              
sma_cross      diaria      0.074  0.096   0.793  -0.185        1.395   
               mensual     0.076  0.098   0.799  -0.185        1.315   
momentum       diaria      0.076  0.099   0.788  -0.185        5.359   
               mensual     0.075  0.100   0.772  -0.185        1.254   
mean_reversion diaria      0.020  0.086   0.278  -0.183       26.916   
               mensual     0.038  0.068   0.582  -0.174        3.854   
vol_target     diaria      0.060  0.065   0.930  -0.143        4.648   
               mensual     0.065  0.069   0.955  -0.136        1.821   

                           cost_drag_yr  
estrategia     frecuencia                
sma_cross      diaria             0.001  
               mensual            0.001  
momentum       diaria             0.003  
               mensual            0.001  
mean_reversion diaria             0.013  
               mensual            0.002  
vol_target     diaria             0.002  
               mensual            0.001

In [4]:
# ¿Cambia la conclusión si los costos fueran altos?
sens = pd.DataFrame({
    (name, label): {bps: sharpe(run_backtest(px, fn(px), cost_bps=bps, start=start)["net"]) for bps in [0, 5, 10, 25]}
    for name in ["mean_reversion", "vol_target", "momentum"]
    for label, fn in [("diaria", STRATEGIES[name]), ("mensual", monthly(STRATEGIES[name]))]
}).T
sens.index.names = ["estrategia", "frecuencia"]; sens.columns.name = "cost_bps"
sens.round(3)

cost_bps                      0      5      10     25
estrategia     frecuencia                            
mean_reversion diaria      0.436  0.278  0.121 -0.350
               mensual     0.610  0.582  0.554  0.469
vol_target     diaria      0.965  0.930  0.894  0.786
               mensual     0.968  0.955  0.941  0.901
momentum       diaria      0.815  0.788  0.761  0.680
               mensual     0.779  0.772  0.766  0.747

### 9.2 Visión desde México: precios en pesos

Un inversionista en México recibe el retorno del ETF **más** el del tipo de cambio: precio en pesos = precio en USD × USD/MXN. Tipo de cambio de Yahoo Finance (`MXN=X`), en caché local en `data/raw/usdmxn.parquet` (no se sube a Git).

In [5]:
fx = download_fx(cache="../data/raw/usdmxn.parquet")
fx_al = fx.reindex(px.index).ffill()
print(f"USD/MXN: {len(fx)} observaciones, {fx.index.min().date()} a {fx.index.max().date()}")
print(f"días de los ETFs sin dato de tipo de cambio ese día (se rellenan con el anterior): {fx.reindex(px.index).isna().sum()}")
print(f"nulos tras rellenar: {fx_al.isna().sum()} | rango: {fx_al.min():.2f} a {fx_al.max():.2f} pesos por dólar")
r_fx = fx_al.pct_change().dropna()
print(f"mayor movimiento diario: {r_fx.abs().max():.1%} ({r_fx.abs().idxmax().date()})")
print(f"USD/MXN: {fx_al.loc[start]:.2f} el {start.date()} -> {fx_al.iloc[-1]:.2f} el {fx_al.index[-1].date()}")
assert fx_al.isna().sum() == 0 and r_fx.abs().max() < 0.15     # sin huecos ni saltos absurdos

USD/MXN: 5466 observaciones, 2005-01-03 a 2025-12-30
días de los ETFs sin dato de tipo de cambio ese día (se rellenan con el anterior): 8
nulos tras rellenar: 0 | rango: 9.87 a 25.34 pesos por dólar
mayor movimiento diario: 9.7% (2008-10-09)
USD/MXN: 10.55 el 2006-01-13 -> 17.97 el 2025-12-30


In [6]:
px_mxn = px.mul(fx_al, axis=0)        # precio en pesos = USD × USDMXN
rows = []
for name, fn in STRATEGIES.items():
    usd = summarize(run_backtest(px, fn(px), 5, start=start))
    mxn = summarize(run_backtest(px_mxn, fn(px_mxn), 5, start=start))
    rows.append({"estrategia": name, "sharpe_usd": usd["sharpe"], "sharpe_mxn": mxn["sharpe"],
                 "cagr_usd": usd["cagr"], "cagr_mxn": mxn["cagr"],
                 "vol_usd": usd["vol"], "vol_mxn": mxn["vol"],
                 "max_dd_usd": usd["max_dd"], "max_dd_mxn": mxn["max_dd"]})
mx = pd.DataFrame(rows).set_index("estrategia")
mx.round(3)

,sharpe_usd,sharpe_mxn,cagr_usd,cagr_mxn,vol_usd,vol_mxn,max_dd_usd,max_dd_mxn
estrategia,,,,,,,,
buy_and_hold,0.788,0.919,0.102,0.132,0.135,0.146,-0.352,-0.316
sma_cross,0.793,0.986,0.074,0.113,0.096,0.116,-0.185,-0.155
momentum,0.788,0.976,0.076,0.116,0.099,0.119,-0.185,-0.181
mean_reversion,0.278,0.258,0.020,0.016,0.086,0.073,-0.183,-0.160
vol_target,0.930,1.040,0.060,0.077,0.065,0.074,-0.143,-0.156


In [7]:
print("Correlación de USD/MXN con cada ETF (retornos diarios):")
print(px.pct_change().corrwith(fx_al.pct_change()).round(2).to_string())

eq = pd.DataFrame({f"{n} ({c})": run_backtest(p, STRATEGIES[n](p), 5, start=start)["equity"]
                   for n in ["buy_and_hold", "vol_target"] for c, p in [("USD", px), ("MXN", px_mxn)]})
ax = eq.plot(logy=True, figsize=(9, 5), title="Buy & hold y volatilidad objetivo: en dólares vs. en pesos")
ax.set_ylabel("Capital (inicio = 1)")
plt.tight_layout()
plt.savefig("../reports/figures/11_usd_vs_mxn.png", dpi=150)

Correlación de USD/MXN con cada ETF (retornos diarios):
Ticker
SPY   -0.38
QQQ   -0.30
TLT    0.13
GLD   -0.09
EEM   -0.45


### 9.3 Panel interactivo

`app/dashboard.py` permite mover la estrategia, el costo por operación y el retraso de ejecución, y ver la curva de capital y las métricas contra buy & hold. Se ejecuta en local:

```bash
pip install streamlit
streamlit run app/dashboard.py
```

No se publica en la nube porque usa datos de Yahoo Finance; para el README basta una captura.

## Conclusiones del Paso 9 (2006-2025, parámetros fijos, 5 bps, delay = 1)

**Rebalanceo mensual: cuánto costaba la reactividad diaria**
- Decidir una vez al mes reduce mucho la rotación sin perder desempeño: momentum pasa de 5.4× a 1.3× al año y volatilidad objetivo de 4.6× a 1.8×, con Sharpe prácticamente igual (0.79 → 0.77 y 0.93 → **0.96**). La reactividad diaria **no valía la pena**: pagaba costos sin mejorar el resultado.
- El caso más claro es la reversión a la media: de 26.9× a 3.9× de rotación, el costo anual baja de 1.3 % a 0.2 % y el Sharpe sube de 0.28 a **0.58**. Con 25 bps la versión diaria cae a −0.35 y la mensual se mantiene en 0.47: la mensual es mucho menos frágil a los costos. Ojo: la versión mensual compra según el z-score del cierre de mes y lo mantiene un mes, así que en la práctica es otra estrategia, no solo la misma más barata.
- **Advertencia de sobreajuste:** las 4 versiones mensuales son intentos nuevos, evaluados solo dentro de muestra y después de ver los resultados diarios (se registran en `EXPERIMENTS.md`; el total pasa de 32 a 36). Ninguna diferencia de esta tabla supera el error estándar del Sharpe (≈ 0.22), así que no cambia el veredicto del Paso 7.

**Visión desde México: precios en pesos**
- El peso pasó de 10.55 a 17.97 por dólar (~2.7 % anual de depreciación). En pesos, buy & hold rinde **13.2 % anual** frente a 10.2 % en dólares, y su Sharpe sube de 0.79 a 0.92.
- El dólar funciona como **cobertura natural** para un inversionista mexicano: USD/MXN tiene correlación negativa con las acciones (−0.38 con SPY, −0.45 con EEM). En las crisis el peso se deprecia justo cuando las acciones caen, así que la caída en pesos es menor (buy & hold −32 % vs. −35 %; en 2008 la diferencia es muy visible en la figura 11).
- Las estrategias de tendencia también mejoran en pesos (cruce de medias 0.99, momentum 0.98, volatilidad objetivo 1.04), pero **el supuesto de efectivo al 0 % es mucho peor en pesos**: el efectivo en México rendiría la tasa de Cetes (aprox. 3-11 % anual en este periodo), así que las estrategias que pasan tiempo fuera del mercado están muy castigadas en este análisis. Usar Cetes (serie del SIE de Banxico) como tasa libre de riesgo es la mejora más importante pendiente para la versión en pesos.

**Panel:** `app/dashboard.py` funciona (probado con el simulador de Streamlit, sin excepciones al cambiar estrategia, costo y retraso).
